In [1]:
# A. Membangun Struktur Direktori HDFS

# Membuat direktori raw dan processed
!hdfs dfs -mkdir -p /user/kyadevi/ecommerce/raw
!hdfs dfs -mkdir -p /user/kyadevi/ecommerce/processed

# Menampilkan struktur direktori yang sudah dibuat
print("=== Struktur Direktori di HDFS ===")
!hdfs dfs -ls /user/kyadevi/ecommerce/
print("=== Isi direktori raw ===")
!hdfs dfs -ls /user/kyadevi/ecommerce/raw
print("=== Isi direktori processed ===")
!hdfs dfs -ls /user/kyadevi/ecommerce/processed





=== Struktur Direktori di HDFS ===
Found 2 items
drwxr-xr-x   - kyadevi supergroup          0 2026-09-09 14:30 /user/kyadevi/ecommerce/processed
drwxr-xr-x   - kyadevi supergroup          0 2026-09-09 14:30 /user/kyadevi/ecommerce/raw
=== Isi direktori raw ===
=== Isi direktori processed ===


In [2]:
# B. Mengunggah Data Mentah ke HDFS

# Upload ketiga file CSV ke direktori raw
!hdfs dfs -put transaksi_magelang.csv /user/kyadevi/ecommerce/raw/
!hdfs dfs -put transaksi_yogyakarta.csv /user/kyadevi/ecommerce/raw/
!hdfs dfs -put transaksi_semarang.csv /user/kyadevi/ecommerce/raw/

# Tampilkan isi direktori raw dengan ukuran file
print("=== Isi direktori raw HDFS ===")
!hdfs dfs -ls -h /user/kyadevi/ecommerce/raw/

# Tampilkan ukuran total
print("\n=== Total ukuran file ===")
!hdfs dfs -du -h /user/kyadevi/ecommerce/raw/

=== Isi direktori raw HDFS ===
Found 3 items
-rw-r--r--   1 kyadevi supergroup     12.0 K 2026-09-09 14:51 /user/kyadevi/ecommerce/raw/transaksi_magelang.csv
-rw-r--r--   1 kyadevi supergroup     11.9 K 2026-09-09 14:51 /user/kyadevi/ecommerce/raw/transaksi_semarang.csv
-rw-r--r--   1 kyadevi supergroup     12.4 K 2026-09-09 14:51 /user/kyadevi/ecommerce/raw/transaksi_yogyakarta.csv

=== Total ukuran file ===
12.0 K  12.0 K  /user/kyadevi/ecommerce/raw/transaksi_magelang.csv
11.9 K  11.9 K  /user/kyadevi/ecommerce/raw/transaksi_semarang.csv
12.4 K  12.4 K  /user/kyadevi/ecommerce/raw/transaksi_yogyakarta.csv


In [3]:
# C. Membaca dan Menggabungkan Data dari HDFS

import pandas as pd
import io

# Fungsi untuk membaca file dari HDFS
def read_csv_from_hdfs(path):
    # Menggunakan perintah hdfs dfs -cat untuk membaca file
    result = !hdfs dfs -cat {path}
    # Menggabungkan hasil menjadi string
    csv_data = '\n'.join(result)
    # Membaca sebagai DataFrame
    return pd.read_csv(io.StringIO(csv_data))

# Baca ketiga file dari HDFS
df_magelang = read_csv_from_hdfs('/user/kyadevi/ecommerce/raw/transaksi_magelang.csv')
df_yogyakarta = read_csv_from_hdfs('/user/kyadevi/ecommerce/raw/transaksi_yogyakarta.csv')
df_semarang = read_csv_from_hdfs('/user/kyadevi/ecommerce/raw/transaksi_semarang.csv')

# Tampilkan informasi masing-masing file
print("=== Data Magelang ===")
print(f"Jumlah baris: {len(df_magelang)}")
print(df_magelang.head(2))

print("\n=== Data Yogyakarta ===")
print(f"Jumlah baris: {len(df_yogyakarta)}")
print(df_yogyakarta.head(2))

print("\n=== Data Semarang ===")
print(f"Jumlah baris: {len(df_semarang)}")
print(df_semarang.head(2))

# Gabungkan ketiga DataFrame
df_gabungan = pd.concat([df_magelang, df_yogyakarta, df_semarang], ignore_index=True)

print(f"\n=== Hasil Gabungan ===")
print(f"Total baris: {len(df_gabungan)}")
print(f"Total kolom: {len(df_gabungan.columns)}")

# Buktikan berisi transaksi dari ketiga kota
print("\n=== Jumlah transaksi per kota ===")
print(df_gabungan['kota'].value_counts())# D. Mengolah dan Upload Hasil ke Direktori Processed

# 1. Tambahkan kolom total_pendapatan
df_gabungan['total_pendapatan'] = df_gabungan['unit_terjual'] * df_gabungan['harga_satuan']

print("=== Data setelah ditambah total_pendapatan ===")
print(df_gabungan[['order_id', 'kota', 'unit_terjual', 'harga_satuan', 'total_pendapatan']].head())

# 2. Buat tabel ringkasan per kota dan per kategori
ringkasan = df_gabungan.groupby(['kota', 'kategori']).agg({
    'total_pendapatan': 'sum',
    'unit_terjual': 'sum',
    'order_id': 'count'
}).rename(columns={'order_id': 'jumlah_transaksi'}).reset_index()

print("\n=== Ringkasan Pendapatan per Kota dan Kategori ===")
print(ringkasan)

# 3. Simpan sebagai CSV di lokal
df_gabungan.to_csv('data_gabungan_bersih.csv', index=False)
ringkasan.to_csv('ringkasan_kota_kategori.csv', index=False)

print("\n✅ File CSV berhasil disimpan di lokal:")
print("- data_gabungan_bersih.csv")
print("- ringkasan_kota_kategori.csv")

# Upload kedua file ke HDFS direktori processed
!hdfs dfs -put data_gabungan_bersih.csv /user/kyadevi/ecommerce/processed/
!hdfs dfs -put ringkasan_kota_kategori.csv /user/kyadevi/ecommerce/processed/

print("\n✅ File berhasil diupload ke HDFS:")
!hdfs dfs -ls -h /user/kyadevi/ecommerce/processed/

=== Data Magelang ===
Jumlah baris: 200
   order_id     tanggal    kategori  unit_terjual  harga_satuan  \
0  MAG-2000  2026-08-12     Fashion             1         50000   
1  MAG-2001  2026-08-18  Elektronik             7         25000   

  metode_pembayaran      kota  
0     Transfer Bank  Magelang  
1               COD  Magelang  

=== Data Yogyakarta ===
Jumlah baris: 200
   order_id     tanggal kategori  unit_terjual  harga_satuan  \
0  YOG-2000  2026-08-20  Fashion             5        250000   
1  YOG-2001  2026-08-24  Fashion             4        150000   

  metode_pembayaran        kota  
0          E-Wallet  Yogyakarta  
1      Kartu Kredit  Yogyakarta  

=== Data Semarang ===
Jumlah baris: 200
   order_id     tanggal                kategori  unit_terjual  harga_satuan  \
0  SEM-2000  2026-08-28              Elektronik             3        100000   
1  SEM-2001  2026-08-29  Kesehatan & Kecantikan             4         25000   

  metode_pembayaran      kota  
0          E-

In [4]:
# D. Mengolah dan Upload Hasil ke Direktori Processed

# 1. Tambahkan kolom total_pendapatan
df_gabungan['total_pendapatan'] = df_gabungan['unit_terjual'] * df_gabungan['harga_satuan']

print("=== Data setelah ditambah total_pendapatan ===")
print(df_gabungan[['order_id', 'kota', 'unit_terjual', 'harga_satuan', 'total_pendapatan']].head())

# 2. Buat tabel ringkasan per kota dan per kategori
ringkasan = df_gabungan.groupby(['kota', 'kategori']).agg({
    'total_pendapatan': 'sum',
    'unit_terjual': 'sum',
    'order_id': 'count'
}).rename(columns={'order_id': 'jumlah_transaksi'}).reset_index()

print("\n=== Ringkasan Pendapatan per Kota dan Kategori ===")
print(ringkasan)

# 3. Simpan sebagai CSV di lokal
df_gabungan.to_csv('data_gabungan_bersih.csv', index=False)
ringkasan.to_csv('ringkasan_kota_kategori.csv', index=False)

print("\n✅ File CSV berhasil disimpan di lokal:")
print("- data_gabungan_bersih.csv")
print("- ringkasan_kota_kategori.csv")

# Upload kedua file ke HDFS direktori processed
!hdfs dfs -put data_gabungan_bersih.csv /user/kyadevi/ecommerce/processed/
!hdfs dfs -put ringkasan_kota_kategori.csv /user/kyadevi/ecommerce/processed/

print("\n✅ File berhasil diupload ke HDFS:")
!hdfs dfs -ls -h /user/kyadevi/ecommerce/processed/

=== Data setelah ditambah total_pendapatan ===
   order_id      kota  unit_terjual  harga_satuan  total_pendapatan
0  MAG-2000  Magelang             1         50000             50000
1  MAG-2001  Magelang             7         25000            175000
2  MAG-2002  Magelang             7         25000            175000
3  MAG-2003  Magelang             6         25000            150000
4  MAG-2004  Magelang             7        100000            700000

=== Ringkasan Pendapatan per Kota dan Kategori ===
          kota                kategori  total_pendapatan  unit_terjual  \
0     Magelang              Elektronik          18775000           197   
1     Magelang                 Fashion          27750000           230   
2     Magelang  Kesehatan & Kecantikan          17375000           138   
3     Magelang       Makanan & Minuman          17525000           144   
4     Magelang            Rumah Tangga          12200000           123   
5     Semarang              Elektronik          2

# Dokumentasi Ecommerce

## A. Struktur Direktori HDFS

![praktikum-bigdata](ecommerce.png)

## B. Isi Folder raw

![praktikum-bigdata](raw.png)

## C. Isi Folder processed

![praktikum-bigdata](processed.png)

## Refleksi: Keuntungan Memisahkan Data Raw dan Processed di HDFS

Menurut saya, memisahkan data mentah (raw) dari data olahan (processed) di HDFS memiliki beberapa keuntungan penting.

Pertama, dari sisi keamanan dan integritas data. Data mentah adalah sumber daya berharga yang tidak boleh diubah atau dihapus secara tidak sengaja. Dengan menyimpannya di folder raw yang terpisah, kita memastikan bahwa data asli selalu tersedia dan dapat diakses kapan saja. Ini sangat penting jika terjadi kesalahan dalam proses pengolahan, karena kita masih memiliki data asli untuk diolah ulang.

Kedua, dari sisi organisasi dan manajemen data. Memisahkan folder membuat struktur data menjadi lebih rapi dan mudah dipahami. Tim data engineer atau data scientist dapat dengan mudah mengetahui mana data yang baru masuk (raw) dan mana yang sudah diproses (processed). Ini memudahkan kolaborasi dalam tim dan mengurangi risiko kesalahan pengambilan data.

Ketiga, dari sisi audit dan tracking. Dengan adanya pemisahan ini, kita dapat melacak alur data dari awal hingga akhir. Proses ETL (Extract, Transform, Load) menjadi lebih transparan dan terdokumentasi dengan baik. Jika ada masalah pada hasil akhir, kita bisa menelusuri kembali ke data mentah untuk menemukan penyebabnya.

Keempat, dari sisi efisiensi pemrosesan. Data processed biasanya sudah dalam format yang lebih ringkas dan siap pakai untuk analisis. Sementara data raw mungkin masih besar dan berat. Dengan memisahkan keduanya, kita bisa mengoptimalkan resource untuk memproses data raw secara batch, sementara data processed siap digunakan untuk query yang lebih cepat.

Secara keseluruhan, pemisahan ini adalah praktik terbaik (best practice) dalam pengelolaan data warehouse dan big data, yang membantu menjaga kualitas, ketertelusuran, dan keberlanjutan sistem data di perusahaan.